# Sesión 07 — Diseño de red de acceso sobre San Isidro

Notebook de diseño construido **fase por fase** siguiendo el flujo
profesional (ver `index.md` y `brainstorming-diseno-red.md`).
El esbozo exploratorio previo vive en `test_scene.ipynb`.

Kernel: `ran-design` (sionna 2.x). Ejecución headless:
`python -m nbconvert --to notebook --execute --inplace design.ipynb`

## Fase 0 — Requisitos del encargo

Todo el diseño se verifica contra estas metas (tabla completa y
justificación en `index.md` §Fase 0). Son constantes del proyecto:
si el negocio las cambia, el diseño se recalcula — por eso viven en
una celda propia al inicio.

In [1]:
# ============ REQUISITOS (Fase 0) — contrato del diseño ============
REQ = {
    # R1 — área de servicio: la escena de San Isidro
    "escena":            "blends/test_scene/untitled.xml",
    "area_km2":          1.32 * 0.83,
    # R2 — cobertura de control
    "rsrp_min_dbm":      -110.0,
    "rsrp_prob":         0.95,
    # R3 — calidad de datos
    "sinr_min_db":       0.0,
    "sinr_prob":         0.90,
    # R4 — throughput de borde (percentil 5)
    "thr_borde_dl_mbps": 50.0,
    "thr_borde_ul_mbps": 5.0,
    # R5 — capacidad agregada en hora cargada
    "capacidad_mbps_km2": 600.0,
    # R7 — espectro licenciado
    "banda":             "n78",
    "fc_hz":             3.5e9,
    "bw_hz":             100e6,
    # R8 — restricciones de despliegue
    "max_sitios":        6,
    "p_tx_dbm_max":      44.0,
}

# La cuenta detrás de R5 (hipótesis de negocio explícitas):
personas_km2, market_share = 25_000, 0.30
gb_mes, f_bh = 10, 0.10
kbps_por_abonado = gb_mes * 8e9 * f_bh / (30 * 3600) / 1e3
demanda = personas_km2 * market_share * kbps_por_abonado / 1e3   # Mbps/km2
print(f"{kbps_por_abonado:.0f} kbps/abonado en hora cargada "
      f"-> demanda {demanda:.0f} Mbps/km2 (requisito: {REQ['capacidad_mbps_km2']:.0f})")
assert demanda <= REQ["capacidad_mbps_km2"], "R5 no cubre la demanda estimada"

74 kbps/abonado en hora cargada -> demanda 556 Mbps/km2 (requisito: 600)


## Fase 1 — Estrategia de espectro

Dos cuentas que fijan el resto del diseño (teoría en `index.md` §Fase 1):

1. **Frecuencia → sitios**: el delta de pérdida entre bandas se convierte en
   factor de radio ($10^{\Delta/10n}$) y de área — la razón física de las
   capas de espectro.
2. **TDD → ancho de banda efectivo**: el patrón DDDSU reparte el tiempo;
   DL y UL no ven los mismos 100 MHz.

In [2]:
import numpy as np

# ---- Cuenta 1: bandas candidatas vs n78 (referencia del encargo) ----
N_PROP = 3.8                       # exponente de propagación urbano denso
bandas_hz = {"n28 (700 MHz)": 0.7e9, "n1 (2.1 GHz)": 2.1e9,
             "n78 (3.5 GHz)": 3.5e9, "n258 (26 GHz)": 26e9}

fc_ref = REQ["fc_hz"]
print(f"{'banda':<15} {'Δpérdida':>9} {'radio rel.':>10} {'sitios rel.':>11}")
for nombre, f in bandas_hz.items():
    delta_db = 20*np.log10(f/fc_ref)          # solo el término de frecuencia
    r_rel = 10**(-delta_db/(10*N_PROP))       # MAPL fijo -> radio relativo
    sitios_rel = 1/r_rel**2                   # sitios ∝ 1/área de celda
    print(f"{nombre:<15} {delta_db:>+7.1f}dB {r_rel:>9.2f}x {sitios_rel:>10.2f}x")

print("\nLección: la banda se paga en sitios — o al revés.")

banda            Δpérdida radio rel. sitios rel.
n28 (700 MHz)     -14.0dB      2.33x       0.18x
n1 (2.1 GHz)       -4.4dB      1.31x       0.58x
n78 (3.5 GHz)      +0.0dB      1.00x       1.00x
n258 (26 GHz)     +17.4dB      0.35x       8.26x

Lección: la banda se paga en sitios — o al revés.


In [3]:
# ---- Cuenta 2: patrón TDD DDDSU -> ancho de banda efectivo ----
# De cada 5 slots: 3 DL + 1 especial (~57% de símbolos DL) + 1 UL
slots = {"DL": 3, "S": 1, "UL": 1}
frac_dl = (slots["DL"] + 0.57*slots["S"]) / sum(slots.values())
frac_ul = slots["UL"] / sum(slots.values())

ESPECTRO = {                       # decisión de la Fase 1 — hereda el diseño
    "patron_tdd":   "DDDSU",
    "frac_dl":      frac_dl,
    "frac_ul":      frac_ul,
    "bw_dl_ef_hz":  REQ["bw_hz"] * frac_dl,
    "bw_ul_ef_hz":  REQ["bw_hz"] * frac_ul,
    "scs_hz":       30e3,          # numerologia mu=1 (S03); CP 2.3 us >> DS 60 ns medido
}
print(f"DL: {frac_dl:.0%} del tiempo -> {ESPECTRO['bw_dl_ef_hz']/1e6:.0f} MHz efectivos")
print(f"UL: {frac_ul:.0%} del tiempo -> {ESPECTRO['bw_ul_ef_hz']/1e6:.0f} MHz efectivos")
print("El UE pierde dos veces: en potencia (23 vs 44 dBm) y en tiempo (1 slot de 5).")

DL: 71% del tiempo -> 71 MHz efectivos
UL: 20% del tiempo -> 20 MHz efectivos
El UE pierde dos veces: en potencia (23 vs 44 dBm) y en tiempo (1 slot de 5).


## Fase 2 — Dimensionamiento por cobertura *(pendiente)*

## Fase 3 — Dimensionamiento por capacidad *(pendiente)*

## Fase 4 — Plan nominal *(pendiente)*

## Fase 5 — Planificación detallada *(pendiente)*

## Fase 6 — Validación *(pendiente)*